# Build and Export Wheels (Kaggle, torch 2.6.0+cu124)
Notebook ini khusus untuk build wheel source `causal-conv1d` dan `mamba-ssm`, lalu export ke `/kaggle/working` agar bisa dipakai ulang tanpa compile ulang.

## Runtime Target
- torch version: `2.6.0+cu124`
- torch cuda: `12.4`
- torch cxx11abi: `False`

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/AneKazek/malesbgt.git"
REPO_BRANCH = "main"

WORKDIR = Path('/kaggle/temp')
VENV = WORKDIR / '.venv'
REPO_DIR = WORKDIR / 'kcv-tts'
WHEEL_DIR = Path('/kaggle/working/wheelhouse')
ARCHIVE = Path('/kaggle/working/wheelhouse_mamba_torch260_cu124.tar.gz')

os.environ['REPO_URL'] = REPO_URL
os.environ['REPO_BRANCH'] = REPO_BRANCH
os.environ['WORKDIR'] = str(WORKDIR)
os.environ['VENV'] = str(VENV)
os.environ['REPO_DIR'] = str(REPO_DIR)
os.environ['WHEEL_DIR'] = str(WHEEL_DIR)
os.environ['ARCHIVE'] = str(ARCHIVE)

print('Config OK')
print('REPO_URL =', REPO_URL)
print('REPO_BRANCH =', REPO_BRANCH)
print('WHEEL_DIR =', WHEEL_DIR)

In [ ]:
%%bash
set -euo pipefail

apt-get update -qq
apt-get install -y python3.11 python3.11-venv python3.11-distutils git > /dev/null

mkdir -p /kaggle/temp
cd /kaggle/temp

python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip setuptools wheel

if [ -d kcv-tts ]; then rm -rf kcv-tts; fi
git clone --depth 1 --branch "${REPO_BRANCH}" "${REPO_URL}" kcv-tts

cd kcv-tts
test -f requirements-py311-torch260-cu124.txt
pip install -r requirements-py311-torch260-cu124.txt
pip install -e . --no-deps

In [ ]:
%%bash
set -euo pipefail

VENV=/kaggle/temp/.venv
WHEEL_DIR=/kaggle/working/wheelhouse
ARCHIVE=/kaggle/working/wheelhouse_mamba_torch260_cu124.tar.gz

echo "=== Step 1: Uninstall old packages ==="
"$VENV/bin/pip" uninstall -y causal-conv1d mamba-ssm 2>/dev/null || true
"$VENV/bin/pip" cache purge || true

echo "=== Step 2: Check torch ABI ==="
source "$VENV/bin/activate"
python - <<'PY'
import torch
print(f"torch version: {torch.__version__}")
print(f"torch cuda: {torch.version.cuda}")
print(f"torch cxx11abi: {torch._C._GLIBCXX_USE_CXX11_ABI}")
PY

echo "=== Step 3: Cleanup old binary artifacts ==="
python - <<'PY'
import site, glob, os, shutil
for d in site.getsitepackages():
    for pattern in ["causal_conv1d*", "mamba_ssm*", "causal_conv1d_cuda*", "selective_scan_cuda*"]:
        for p in glob.glob(os.path.join(d, pattern)):
            try:
                if os.path.isdir(p):
                    shutil.rmtree(p, ignore_errors=True)
                else:
                    os.remove(p)
                print(f"removed: {p}")
            except Exception:
                pass
PY

echo "=== Step 4: Build wheels from source ==="
mkdir -p "$WHEEL_DIR"
export CXXFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"
export CFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"
export LDFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"
export LD_LIBRARY_PATH="/usr/local/cuda/lib64:/usr/local/cuda-12.4/lib64:${LD_LIBRARY_PATH:-}"

echo "Building wheel causal-conv1d==1.6.1 ..."
pip wheel --no-deps --no-cache-dir --no-build-isolation --no-binary=:all: --wheel-dir "$WHEEL_DIR" causal-conv1d==1.6.1

echo "Building wheel mamba-ssm==2.3.1 ..."
pip wheel --no-deps --no-cache-dir --no-build-isolation --no-binary=:all: --wheel-dir "$WHEEL_DIR" mamba-ssm==2.3.1

echo "=== Step 5: Install from local wheels ==="
ls -lh "$WHEEL_DIR"
pip install --no-deps --no-cache-dir --force-reinstall "$WHEEL_DIR"/causal_conv1d-*.whl "$WHEEL_DIR"/mamba_ssm-*.whl

echo "=== Step 6: Verify import ==="
python - <<'PY'
import sys, torch
try:
    import causal_conv1d
    print('OK causal_conv1d imported')
except Exception as e:
    print(f'FAIL causal_conv1d import: {e}')
    sys.exit(1)

try:
    import mamba_ssm
    print('OK mamba_ssm imported')
except Exception as e:
    print(f'FAIL mamba_ssm import: {e}')
    sys.exit(1)

print(f'OK torch ABI: {torch._C._GLIBCXX_USE_CXX11_ABI}')
PY

echo "=== Step 7: Pack wheels for download ==="
tar -czf "$ARCHIVE" -C /kaggle/working wheelhouse
ls -lh "$ARCHIVE"
echo "Done. Download wheelhouse folder or tar.gz from /kaggle/working."

In [ ]:
%%bash
set -euo pipefail

# Optional: next run without compile.
# Attach your uploaded wheel dataset and set WHEEL_SRC_DIR accordingly.
VENV=/kaggle/temp/.venv
WHEEL_SRC_DIR="${WHEEL_SRC_DIR:-/kaggle/input/your-wheelhouse-dataset/wheelhouse}"

if [ ! -d "$WHEEL_SRC_DIR" ]; then
  echo "Wheel source directory not found: $WHEEL_SRC_DIR"
  echo "Set WHEEL_SRC_DIR to your mounted Kaggle dataset wheelhouse path."
  exit 0
fi

source "$VENV/bin/activate"
pip install --no-deps --no-cache-dir --force-reinstall \
  "$WHEEL_SRC_DIR"/causal_conv1d-*.whl \
  "$WHEEL_SRC_DIR"/mamba_ssm-*.whl

python - <<'PY'
import torch, causal_conv1d, mamba_ssm
print('OK torch', torch.__version__, 'cuda', torch.version.cuda, 'abi', torch._C._GLIBCXX_USE_CXX11_ABI)
print('OK causal_conv1d', getattr(causal_conv1d, '__version__', 'n/a'))
print('OK mamba_ssm', getattr(mamba_ssm, '__version__', 'n/a'))
PY